# Data summary (read-only)

Reads the JSON files in `../data/` and prints counts, **usability** (total vs usable),
keyword coverage, and ECHR breakdowns. No plots, no files written, nothing modified.
Defensive throughout (`.get`, never assumes a key); reports expected-but-missing fields.

## 1. Inspect every JSON file in data/
Filename · top-level type · record count · field names (and whether fields are consistent
across records).

In [ ]:
import json, glob, os, re
from collections import Counter

DATA_DIR = "../data"


def load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f), None
    except Exception as e:
        return None, str(e)


print("=" * 72)
print("JSON FILES IN", os.path.abspath(DATA_DIR))
print("=" * 72)
for path in sorted(glob.glob(os.path.join(DATA_DIR, "*.json"))):
    name = os.path.basename(path)
    data, err = load_json(path)
    if err:
        print(f"\n{name}\n   ERROR loading: {err}")
        continue
    if isinstance(data, list):
        dict_recs = [r for r in data if isinstance(r, dict)]
        print(f"\n{name}\n   top-level: list | records: {len(data)}")
        if dict_recs:
            keysets = [set(r.keys()) for r in dict_recs]
            union = set().union(*keysets)
            common = set.intersection(*keysets)
            print(f"   fields: {len(union)} | consistent across records: {union == common}")
            print(f"     {sorted(union)}")
            if union != common:
                print(f"     NOT on every record: {sorted(union - common)}")
        else:
            print("   (empty or no dict records)")
    elif isinstance(data, dict):
        print(f"\n{name}\n   top-level: dict | keys: {sorted(data.keys())}")
    else:
        print(f"\n{name}\n   top-level: {type(data).__name__}")

JSON FILES IN /Users/maksimsmirnov/Desktop/thesis/data

derstandard_comments.json
   top-level: list | records: 0
   (empty or no dict records)

derstandard_scrape_log.json
   top-level: list | records: 6
   fields: 6 | consistent across records: True
     ['article_id', 'article_url', 'error', 'n_comments', 'ok', 'scraped_at']

echr_llm_extraction_checkpoint.json
   top-level: dict | keys: ['001-150216', '001-151139', '001-154162', '001-155910', '001-156522', '001-157293', '001-159300', '001-159844', '001-160212', '001-161002', '001-161462', '001-164917', '001-164923', '001-166714', '001-168066', '001-170860', '001-172077', '001-172561', '001-173800', '001-174422', '001-174699', '001-177079', '001-177080', '001-178097', '001-179421', '001-180505', '001-181927', '001-182208', '001-182213', '001-182754', '001-184281', '001-186439', '001-187532', '001-187793', '001-187931', '001-189774', '001-191488', '001-192205', '001-194107', '001-194531', '001-194735', '001-195909', '001-196416', '00

## 2. ECHR — usability, keywords, breakdowns
Usable = non-empty `full_text` (whitespace stripped). All breakdowns are on the **usable** set.

In [ ]:
ECHR_PATH = os.path.join(DATA_DIR, "echr_parental_alienation.json")
echr, err = load_json(ECHR_PATH)
if err or not isinstance(echr, list):
    print(f"ECHR file unavailable ({err or 'not a list'}) — skipping ECHR summary")
    echr = []

# report expected-but-missing fields rather than crashing later
EXPECTED = ["full_text", "matched_keywords", "respondent", "judgementdate",
            "itemid", "appno", "ecli", "referencedate"]
if echr:
    seen = set().union(*[set(r.keys()) for r in echr[:100] if isinstance(r, dict)])
    missing = [f for f in EXPECTED if f not in seen]
    if missing:
        print("NOTE: expected ECHR fields not present:", missing, "\n")

total = len(echr)
usable = [r for r in echr if (r.get("full_text") or "").strip()]
print("=" * 72)
print("ECHR SUMMARY")
print("=" * 72)
print("USABILITY")
print(f"  total records         : {total}")
print(f"  usable (full_text)    : {len(usable)}")
print(f"  unusable (empty text) : {total - len(usable)}")

# keyword coverage on usable set
kw_present = sum(1 for r in usable if r.get("matched_keywords"))
kwc = Counter(k for r in usable for k in (r.get("matched_keywords") or []))
print("\nKEYWORDS (usable set)")
print(f"  usable with non-empty matched_keywords: {kw_present}/{len(usable)}")
for k, c in kwc.most_common():
    print(f"     {c:5d}  {k}")

# cases per respondent country (usable), descending
print("\nCASES PER RESPONDENT COUNTRY (usable, desc)")
for country, c in Counter((r.get("respondent") or "(none)") for r in usable).most_common():
    print(f"     {c:5d}  {country}")


def echr_year(r):
    # primary: judgementdate timestamp "DD/MM/YYYY 00:00:00"
    raw = (r.get("judgementdate") or "").strip()
    if raw:
        parts = raw.split("/")
        if len(parts) >= 3:
            try:
                return int(parts[2].split()[0][:4])
            except ValueError:
                pass
    # fallback date field: ECLI encodes the year (ECLI:CE:ECHR:YYYY:...)
    m = re.match(r"ECLI:CE:ECHR:(\d{4}):", r.get("ecli", "") or "")
    if m:
        return int(m.group(1))
    # fallback date field: referencedate (if populated)
    rd = (r.get("referencedate") or "").strip()
    if rd:
        mm = re.search(r"(\d{4})", rd)
        if mm:
            return int(mm.group(1))
    return None


years, noyear = Counter(), 0
for r in usable:
    y = echr_year(r)
    if y is None:
        noyear += 1
    else:
        years[y] += 1
print("\nCASES PER YEAR (usable; judgementdate, else ECLI/referencedate)")
for y in sorted(years):
    print(f"     {y}: {years[y]}")
print(f"     no parseable year: {noyear}")

# duplicate / inflation check
ids = [r.get("itemid") for r in echr if r.get("itemid")]
appnos = [r.get("appno") for r in echr if r.get("appno")]
print("\nDISTINCT-vs-RAW (is anything inflating totals?)")
print(f"  raw records      : {total}")
print(f"  distinct itemid  : {len(set(ids))}   (records carrying itemid: {len(ids)})")
print(f"  distinct appno   : {len(set(appnos))}   (records carrying appno: {len(appnos)})")

ECHR SUMMARY
USABILITY
  total records         : 1116
  usable (full_text)    : 1116
  unusable (empty text) : 0

KEYWORDS (usable set)
  usable with non-empty matched_keywords: 1116/1116
       748  best interests of the child
       554  contact rights
       227  child welfare
        41  parental alienation

CASES PER RESPONDENT COUNTRY (usable, desc)
       116  NOR
        99  RUS
        76  DEU
        73  POL
        67  ROU
        53  FIN
        49  GBR
        44  UKR
        43  HRV
        42  HUN
        41  BGR
        38  SWE
        29  AUT
        26  SRB
        24  CZE
        20  NLD
        18  SVK
        18  SVN
        18  LTU
        18  GRC
        17  FRA
        17  ITA
        17  LVA
        16  CHE
        15  PRT
        13  ESP
        11  TUR
        10  MDA
        10  EST
        10  MKD
         9  MLT
         8  DNK
         6  GEO
         6  ARM
         5  BEL
         5  AZE
         5  CYP
         4  ISL
         3  MNE
         3  IRL
  

## 3. RIS — usability, keywords
Usable = the Rechtssatz **principle** is non-empty. Principle = the text after the
`Rechtssatz` header and before `Entscheidungstexte` in `full_text` (RIS has no dedicated
principle field). Records without that layout (the `Text` full-decisions) yield no principle
and are listed as unusable.

In [ ]:
RIS_PATH = os.path.join(DATA_DIR, "ris_parental_alienation.json")
ris, err = load_json(RIS_PATH)
if err or not isinstance(ris, list):
    print(f"RIS file unavailable ({err or 'not a list'}) — skipping RIS summary")
    ris = []

if ris:
    seen = set().union(*[set(r.keys()) for r in ris if isinstance(r, dict)])
    for f in ["full_text", "matched_keywords", "id", "dokumenttyp"]:
        if f not in seen:
            print("NOTE: expected RIS field not present:", f)


def ris_principle(full_text):
    # text between the 'Rechtssatz' label line and the 'Entscheidungstexte' label line
    lines = (full_text or "").split("\n")

    def find(label):
        for i, l in enumerate(lines):
            if l.strip() == label:
                return i
        return -1

    i_rs = find("Rechtssatz")
    if i_rs == -1:
        return ""
    i_et = find("Entscheidungstexte")
    i_ecli = find("European Case Law Identifier")
    end = i_et if i_et != -1 else (i_ecli if i_ecli != -1 else len(lines))
    return "\n".join(lines[i_rs + 1:end]).strip()


total = len(ris)
usable = [r for r in ris if ris_principle(r.get("full_text", "")).strip()]
unusable = [r for r in ris if not ris_principle(r.get("full_text", "")).strip()]
print("=" * 72)
print("RIS SUMMARY")
print("=" * 72)
print("USABILITY")
print(f"  total records              : {total}")
print(f"  usable (principle present) : {len(usable)}")
print(f"  unusable (no principle)    : {len(unusable)}")
print(f"  usable by dokumenttyp      : {dict(Counter(r.get('dokumenttyp') for r in usable))}")
print(f"  unusable by dokumenttyp    : {dict(Counter(r.get('dokumenttyp') for r in unusable))}")

print(f"\nUNUSABLE IDS ({len(unusable)})")
for r in unusable:
    print(f"     {r.get('id') or r.get('stable_id') or '(no id)'}")

kw_present = sum(1 for r in usable if r.get("matched_keywords"))
kwc = Counter(k for r in usable for k in (r.get("matched_keywords") or []))
print("\nKEYWORDS (usable set)")
print(f"  usable with non-empty matched_keywords: {kw_present}/{len(usable)}")
for k, c in kwc.most_common():
    print(f"     {c:5d}  {k}")

RIS SUMMARY
USABILITY
  total records              : 548
  usable (principle present) : 38
  unusable (no principle)    : 510
  usable by dokumenttyp      : {'Rechtssatz': 38}
  unusable by dokumenttyp    : {'Text': 510}

UNUSABLE IDS (510)
     JJT_20060405_OGH0002_0130OS00007_06P0000_000
     JJT_20101118_OGH0002_0130OS00052_10M0000_000
     JJT_20070503_OGH0002_0120OS00038_07S0000_000
     JJT_20090513_OGH0002_0150OS00044_09Y0000_000
     JJT_20131217_OGH0002_0140OS00162_13Z0000_000
     JJT_20140826_OGH0002_0110OS00044_14A0000_000
     JJT_20151117_OGH0002_0140OS00098_15S0000_000
     JJT_20020711_AUSL000_000BSW28957_9500000_000
     JJT_20030612_AUSL000_000BSW35968_9700000_000
     JJT_20070911_AUSL000_000BSW27527_0300000_000
     JJT_20140716_AUSL000_000BSW37359_0900000_000
     JJT_20150310_AUSL000_000BSW14793_0800000_000
     JJT_20170406_AUSL000_000BSW79885_1200000_000
     JJT_20121016_OGH0002_0140OS00081_12M0000_000
     JJT_20140401_OGH0002_0140OS00015_14H0000_000
     JJT_

## Notes
- ECHR usable = non-empty `full_text`; RIS usable = non-empty extracted principle (the
  `Text` full-decision records have no principle layout, so they count as unusable here).
- Read-only: no data file or other notebook was modified, and nothing was written to disk.